# SignLearn-06 — Mirror Augmentation and Camera Calibration

This notebook improves orientation robustness **without loading or using WLASL**. Model development uses only the frozen Google signer split. WLASL remains a locked external test set.

Required Kaggle inputs:
1. `mvp50_temporal_48_frames.npz` from SignLearn-03.
2. `signlearn_model.keras` and the JSON files from the Keras deployment bundle, or `signlearn_deployment_bundle.zip`.

Scientific rule: do not change this notebook after viewing final WLASL results. Previously inspected WLASL clips must be treated as pilot data, not final test data.

In [ ]:
from pathlib import Path
from IPython.display import display
import json
import shutil
import time
import warnings
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, top_k_accuracy_score

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', context='notebook')
SEED = 42
BATCH_SIZE = 128
FINE_TUNE_EPOCHS = 18
AUGMENT_PROBABILITY = 0.50
MIRROR_MODE = 'reflect_and_swap_hands'  # or 'reflect_only'
TARGET_ACCEPTED_ACCURACY = 0.85
MINIMUM_COVERAGE = 0.10

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

KAGGLE_INPUT = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working/signlearn_06')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
print('Output:', OUTPUT_DIR)

## 1. Locate the previous SignLearn artifacts

The code searches all attached Kaggle inputs. If only the deployment ZIP is attached, it extracts it under `/kaggle/working`. No WLASL path is searched.

In [ ]:
EXTRACTED_BUNDLE = OUTPUT_DIR / 'input_bundle'
zip_candidates = list(KAGGLE_INPUT.rglob('signlearn_deployment_bundle.zip'))
if zip_candidates and not EXTRACTED_BUNDLE.exists():
    with zipfile.ZipFile(zip_candidates[0]) as archive:
        archive.extractall(EXTRACTED_BUNDLE)
    print('Extracted deployment bundle:', zip_candidates[0])

search_roots = [KAGGLE_INPUT, EXTRACTED_BUNDLE]

def find_artifact(names, required=True):
    names = [names] if isinstance(names, str) else list(names)
    for name in names:
        for root in search_roots:
            if root.exists():
                matches = list(root.rglob(name))
                if matches:
                    return matches[0]
    if required:
        raise FileNotFoundError(
            f'Missing {names}. Attach the SignLearn-03 output cache and Keras deployment bundle.'
        )
    return None

CACHE_PATH = find_artifact('mvp50_temporal_48_frames.npz')
MODEL_PATH = find_artifact(['signlearn_model.keras', 'best_temporal_bilstm.keras'])
LABEL_MAP_PATH = find_artifact('label_map.json')
MANIFEST_PATH = find_artifact('deployment_manifest.json', required=False)
TEMPORAL_CONFIG_PATH = find_artifact('temporal_configuration.json')
FEATURE_NAMES_PATH = find_artifact('frame_feature_names.json')

print('Temporal cache:', CACHE_PATH)
print('Starting model:', MODEL_PATH)
print('Label map:', LABEL_MAP_PATH)
assert 'wlasl' not in str(CACHE_PATH).lower(), 'WLASL must not be used for development.'

In [ ]:
cache = np.load(CACHE_PATH, allow_pickle=False)
X = cache['X'].astype(np.float32)
y = cache['y'].astype(np.int32)
splits = cache['splits'].astype(str)
participant_ids = cache['participant_ids']
sequence_ids = cache['sequence_ids']

with LABEL_MAP_PATH.open('r', encoding='utf-8') as stream:
    label_map = json.load(stream)
index_to_sign = {int(index): sign for sign, index in label_map.items()}

train_mask = splits == 'train'
validation_mask = splits == 'validation'
test_mask = splits == 'test'
X_train, y_train = X[train_mask], y[train_mask]
X_validation, y_validation = X[validation_mask], y[validation_mask]
X_test, y_test = X[test_mask], y[test_mask]

participant_sets = {
    split_name: set(participant_ids[splits == split_name].tolist())
    for split_name in ['train', 'validation', 'test']
}
assert participant_sets['train'].isdisjoint(participant_sets['validation'])
assert participant_sets['train'].isdisjoint(participant_sets['test'])
assert participant_sets['validation'].isdisjoint(participant_sets['test'])
assert X.shape[1:] == (48, 189)
assert sorted(label_map.values()) == list(range(50))

split_summary = pd.DataFrame({
    'sequences': [train_mask.sum(), validation_mask.sum(), test_mask.sum()],
    'signers': [len(participant_sets[name]) for name in ['train', 'validation', 'test']],
}, index=['train', 'validation', 'test'])
display(split_summary)
print('The test tensors are loaded but will not be evaluated until the policy is frozen.')

## 2. Define orientation transformations

The 189 features contain 63 left-hand coordinates, 63 right-hand coordinates, 18 pose coordinates, 40 lip coordinates, and five detection indicators. Because coordinates are shoulder-centered, horizontal reflection negates every x-coordinate.

Two diagnostic conventions are provided:
- `reflect_only`: image-plane reflection while retaining anatomical feature slots.
- `reflect_and_swap_hands`: reflection plus hand-channel and hand-detection swapping, approximating extractors that reverse hand channels after a mirrored input.

Pose and lip identities are retained because they are anatomical MediaPipe indices. Raw-video mirroring and re-extraction is preferable when original videos are available.

In [ ]:
FEATURE_COUNT = 189
LEFT_HAND = slice(0, 63)
RIGHT_HAND = slice(63, 126)
POSE_START, POSE_END = 126, 144
LIP_START, LIP_END = 144, 184
LEFT_DETECTION, RIGHT_DETECTION = 184, 185

reflection_sign = np.ones(FEATURE_COUNT, dtype=np.float32)
reflection_sign[0:63:3] = -1.0
reflection_sign[63:126:3] = -1.0
reflection_sign[POSE_START:POSE_END:2] = -1.0
reflection_sign[LIP_START:LIP_END:2] = -1.0

identity_permutation = np.arange(FEATURE_COUNT, dtype=np.int32)
swap_hand_permutation = identity_permutation.copy()
swap_hand_permutation[LEFT_HAND] = np.arange(63, 126)
swap_hand_permutation[RIGHT_HAND] = np.arange(0, 63)
swap_hand_permutation[LEFT_DETECTION] = RIGHT_DETECTION
swap_hand_permutation[RIGHT_DETECTION] = LEFT_DETECTION

def mirror_features(features, mode='reflect_only'):
    features = np.asarray(features, dtype=np.float32)
    if mode == 'reflect_only':
        permutation = identity_permutation
    elif mode == 'reflect_and_swap_hands':
        permutation = swap_hand_permutation
    else:
        raise ValueError(f'Unknown mirror mode: {mode}')
    return features[..., permutation] * reflection_sign

for mode in ['reflect_only', 'reflect_and_swap_hands']:
    sample = X_validation[:16]
    twice = mirror_features(mirror_features(sample, mode), mode)
    np.testing.assert_allclose(twice, sample, atol=1e-6)
    assert mirror_features(sample, mode).shape == sample.shape
print('Mirror transformations passed shape and involution tests.')

## 3. Validation-only orientation audit

This cell measures how the original model reacts to synthetic reflections. It does not access the Google test labels or WLASL. Low mirror accuracy or low prediction agreement confirms orientation sensitivity.

In [ ]:
def probability_metrics(true_labels, probabilities):
    predictions = probabilities.argmax(axis=1)
    return {
        'accuracy': float(accuracy_score(true_labels, predictions)),
        'macro_f1': float(f1_score(true_labels, predictions, average='macro')),
        'top5_accuracy': float(top_k_accuracy_score(
            true_labels, probabilities, k=5, labels=np.arange(50)
        )),
    }

def predict_array(model, features):
    dataset = tf.data.Dataset.from_tensor_slices(features).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return model.predict(dataset, verbose=0)

baseline_model = tf.keras.models.load_model(MODEL_PATH, compile=False)
baseline_original_probabilities = predict_array(baseline_model, X_validation)
baseline_original_predictions = baseline_original_probabilities.argmax(axis=1)
audit_rows = []

original_metrics = probability_metrics(y_validation, baseline_original_probabilities)
audit_rows.append({'view': 'original', **original_metrics, 'agreement_with_original': 1.0})
for mode in ['reflect_only', 'reflect_and_swap_hands']:
    probabilities = predict_array(baseline_model, mirror_features(X_validation, mode))
    predictions = probabilities.argmax(axis=1)
    audit_rows.append({
        'view': mode,
        **probability_metrics(y_validation, probabilities),
        'agreement_with_original': float((predictions == baseline_original_predictions).mean()),
    })

baseline_orientation_audit = pd.DataFrame(audit_rows).set_index('view')
display(baseline_orientation_audit.style.format('{:.4f}'))
baseline_orientation_audit.to_csv(OUTPUT_DIR / 'baseline_orientation_audit.csv')
print('Configured augmentation mode:', MIRROR_MODE)

## 4. Fine-tune with mirror augmentation

Each training sequence is mirrored with probability 0.5. Validation data are never augmented during training. The existing model is fine-tuned with a low learning rate so it retains the original signer-independent representation.

In [ ]:
tf_reflection_sign = tf.constant(reflection_sign, dtype=tf.float32)
selected_permutation = (
    swap_hand_permutation if MIRROR_MODE == 'reflect_and_swap_hands' else identity_permutation
)
tf_selected_permutation = tf.constant(selected_permutation, dtype=tf.int32)

def augment_training_example(features, label):
    features = tf.cast(features, tf.float32)
    mirrored = tf.gather(features, tf_selected_permutation, axis=-1) * tf_reflection_sign
    use_mirror = tf.random.uniform([], seed=SEED) < AUGMENT_PROBABILITY
    return tf.cond(use_mirror, lambda: mirrored, lambda: features), label

AUTOTUNE = tf.data.AUTOTUNE
train_dataset = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(len(X_train), seed=SEED, reshuffle_each_iteration=True)
    .map(augment_training_example, num_parallel_calls=AUTOTUNE, deterministic=True)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
validation_dataset = (
    tf.data.Dataset.from_tensor_slices((X_validation, y_validation))
    .batch(BATCH_SIZE).prefetch(AUTOTUNE)
)

augmented_model = tf.keras.models.load_model(MODEL_PATH, compile=False)
augmented_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy'),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name='top5_accuracy'),
    ],
)
BEST_MODEL_PATH = OUTPUT_DIR / 'best_mirror_robust_temporal.keras'
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(BEST_MODEL_PATH, monitor='val_loss', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
]
history = augmented_model.fit(
    train_dataset, validation_data=validation_dataset, epochs=FINE_TUNE_EPOCHS,
    callbacks=callbacks, verbose=2
)
pd.DataFrame(history.history).to_csv(OUTPUT_DIR / 'mirror_finetuning_history.csv', index=False)

In [ ]:
history_frame = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(history_frame['loss'], label='train')
axes[0].plot(history_frame['val_loss'], label='validation')
axes[0].set(title='Mirror fine-tuning loss', xlabel='Epoch', ylabel='Loss')
axes[1].plot(history_frame['accuracy'], label='train')
axes[1].plot(history_frame['val_accuracy'], label='validation')
axes[1].set(title='Mirror fine-tuning accuracy', xlabel='Epoch', ylabel='Accuracy')
for axis in axes:
    axis.legend()
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'mirror_finetuning_curves.png', dpi=160, bbox_inches='tight')
plt.show()

## 5. Select the orientation and confidence policy using validation only

We compare a calibrated single view with a dual-view probability average. The dual view is only a diagnostic until the application is updated to extract landmarks from both raw orientations.

In [ ]:
augmented_model = tf.keras.models.load_model(BEST_MODEL_PATH, compile=False)
validation_original = predict_array(augmented_model, X_validation)
validation_mirrored = predict_array(augmented_model, mirror_features(X_validation, MIRROR_MODE))
validation_dual = (validation_original + validation_mirrored) / 2.0

candidate_probabilities = {
    'calibrated_single_view': validation_original,
    'synthetic_mirrored_view': validation_mirrored,
    'dual_view_average': validation_dual,
}
candidate_rows = []
for policy_name, probabilities in candidate_probabilities.items():
    candidate_rows.append({'policy': policy_name, **probability_metrics(y_validation, probabilities)})
validation_policy_table = pd.DataFrame(candidate_rows).set_index('policy')
display(validation_policy_table.style.format('{:.4f}'))
validation_policy_table.to_csv(OUTPUT_DIR / 'validation_orientation_policy.csv')

# Deploy the calibrated single view by default. Change this only from validation evidence, never WLASL.
DEPLOYMENT_POLICY = 'calibrated_single_view'
validation_policy_probabilities = candidate_probabilities[DEPLOYMENT_POLICY]

def selective_curve(probabilities, true_labels, thresholds):
    predictions = probabilities.argmax(axis=1)
    confidences = probabilities.max(axis=1)
    rows = []
    for threshold in thresholds:
        accepted = confidences >= threshold
        rows.append({
            'threshold': float(threshold),
            'coverage': float(accepted.mean()),
            'accepted_accuracy': (
                float((predictions[accepted] == true_labels[accepted]).mean())
                if accepted.any() else np.nan
            ),
            'accepted_count': int(accepted.sum()),
        })
    return pd.DataFrame(rows)

validation_selective = selective_curve(
    validation_policy_probabilities, y_validation, np.arange(0.0, 1.0, 0.01)
)
eligible = validation_selective[
    (validation_selective['accepted_accuracy'] >= TARGET_ACCEPTED_ACCURACY)
    & (validation_selective['coverage'] >= MINIMUM_COVERAGE)
]
if eligible.empty:
    chosen_row = validation_selective.sort_values(
        ['accepted_accuracy', 'coverage'], ascending=False
    ).iloc[0]
    print('Warning: target accepted accuracy was not reached; selected the best validation row.')
else:
    chosen_row = eligible.sort_values('threshold').iloc[0]

CONFIDENCE_THRESHOLD = float(chosen_row['threshold'])
print('Frozen deployment policy:', DEPLOYMENT_POLICY)
print('Frozen confidence threshold:', CONFIDENCE_THRESHOLD)
display(chosen_row.to_frame('selected_validation_policy'))
validation_selective.to_csv(OUTPUT_DIR / 'validation_selective_curve.csv', index=False)

## 6. Freeze first, then evaluate once on Google test signers

No model, mirror mode, deployment policy, or confidence threshold may be changed based on the following results. WLASL is still untouched.

In [ ]:
test_original = predict_array(augmented_model, X_test)
if DEPLOYMENT_POLICY == 'dual_view_average':
    test_mirrored = predict_array(augmented_model, mirror_features(X_test, MIRROR_MODE))
    test_policy_probabilities = (test_original + test_mirrored) / 2.0
else:
    test_policy_probabilities = test_original

test_metrics = probability_metrics(y_test, test_policy_probabilities)
test_predictions = test_policy_probabilities.argmax(axis=1)
test_confidences = test_policy_probabilities.max(axis=1)
accepted = test_confidences >= CONFIDENCE_THRESHOLD
test_metrics.update({
    'coverage_at_frozen_threshold': float(accepted.mean()),
    'accepted_accuracy_at_frozen_threshold': (
        float((test_predictions[accepted] == y_test[accepted]).mean()) if accepted.any() else np.nan
    ),
    'accepted_count': int(accepted.sum()),
})
display(pd.Series(test_metrics, name='frozen_google_test').to_frame().style.format('{:.4f}'))

test_results = pd.DataFrame({
    'sequence_id': sequence_ids[test_mask],
    'participant_id': participant_ids[test_mask],
    'true_label': y_test,
    'true_sign': [index_to_sign[int(value)] for value in y_test],
    'predicted_label': test_predictions,
    'predicted_sign': [index_to_sign[int(value)] for value in test_predictions],
    'confidence': test_confidences,
    'accepted': accepted,
})
test_results.to_csv(OUTPUT_DIR / 'frozen_google_test_predictions.csv', index=False)

In [ ]:
confusion = confusion_matrix(y_test, test_predictions, labels=np.arange(50), normalize='true')
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(confusion, cmap='mako', vmin=0, vmax=1, xticklabels=False, yticklabels=False, ax=ax)
ax.set(title='Mirror-robust model: normalized Google test confusion matrix', xlabel='Predicted', ylabel='True')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'mirror_robust_test_confusion.png', dpi=160, bbox_inches='tight')
plt.show()

## 7. Export the updated model and frozen camera policy

The recommended application behaviour is still one orientation per camera session. Ask the user to raise their right hand during setup, determine whether the saved recording is mirrored, and retain that setting. Do not select orientation using the expected sign label.

In [ ]:
EXPORT_DIR = OUTPUT_DIR / 'signlearn_mirror_robust_bundle'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_MODEL_PATH = EXPORT_DIR / 'signlearn_model.keras'
augmented_model.save(EXPORT_MODEL_PATH)

camera_policy = {
    'development_data': 'Google ASL Signs only; WLASL not used',
    'mirror_augmentation_mode': MIRROR_MODE,
    'mirror_augmentation_probability': AUGMENT_PROBABILITY,
    'deployment_policy': DEPLOYMENT_POLICY,
    'camera_orientation': 'calibrate once per camera session using a right-hand-raise prompt',
    'default_flip': False,
    'confidence_threshold': CONFIDENCE_THRESHOLD,
    'threshold_selected_on': 'Google validation signers only',
    'target_accepted_accuracy': TARGET_ACCEPTED_ACCURACY,
    'validation_selected_row': {
        key: (int(value) if key == 'accepted_count' else float(value))
        for key, value in chosen_row.to_dict().items()
    },
    'frozen_google_test_metrics': test_metrics,
    'external_test_status': 'WLASL remains locked and unevaluated in this notebook',
}
with (EXPORT_DIR / 'camera_orientation_policy.json').open('w', encoding='utf-8') as stream:
    json.dump(camera_policy, stream, indent=2)
with (EXPORT_DIR / 'label_map.json').open('w', encoding='utf-8') as stream:
    json.dump(label_map, stream, indent=2)
with (EXPORT_DIR / 'mirror_validation_audit.json').open('w', encoding='utf-8') as stream:
    json.dump({
        'baseline': baseline_orientation_audit.reset_index().to_dict(orient='records'),
        'fine_tuned': validation_policy_table.reset_index().to_dict(orient='records'),
    }, stream, indent=2)

shutil.copy2(TEMPORAL_CONFIG_PATH, EXPORT_DIR / 'temporal_configuration.json')
shutil.copy2(FEATURE_NAMES_PATH, EXPORT_DIR / 'frame_feature_names.json')
if MANIFEST_PATH is not None:
    with MANIFEST_PATH.open('r', encoding='utf-8') as stream:
        deployment_manifest = json.load(stream)
    shutil.copy2(MANIFEST_PATH, EXPORT_DIR / 'original_deployment_manifest.json')
else:
    deployment_manifest = {}
deployment_manifest.update({
    'project': 'SignLearn ASL Practice Assistant — mirror-robust update',
    'inference_backend': 'tensorflow_keras',
    'input_shape': [1, 48, 189],
    'input_dtype': 'float32',
    'class_count': 50,
    'confidence_threshold': CONFIDENCE_THRESHOLD,
    'confidence_policy_reason': 'Selected on Google validation signers after mirror fine-tuning.',
    'mirror_augmentation_mode': MIRROR_MODE,
    'mirror_augmentation_probability': AUGMENT_PROBABILITY,
    'frozen_google_test_metrics': test_metrics,
    'external_test_status': 'WLASL not used during model development',
})
with (EXPORT_DIR / 'deployment_manifest.json').open('w', encoding='utf-8') as stream:
    json.dump(deployment_manifest, stream, indent=2)
archive_path = shutil.make_archive(
    str(OUTPUT_DIR / 'signlearn_mirror_robust_bundle'), 'zip', root_dir=EXPORT_DIR
)
print('Exported:', archive_path)
for artifact in sorted(EXPORT_DIR.iterdir()):
    print(f'  {artifact.name}: {artifact.stat().st_size:,} bytes')

## Interpretation checklist

- Compare original validation macro F1 before and after fine-tuning.
- Check whether mirrored-view macro F1 and agreement improved.
- Reject the update if original performance collapses, even if mirror robustness improves.
- Do not select a policy from WLASL performance.
- After the model and protocol are frozen, run a separate SignLearn-07 notebook for one-time WLASL external evaluation.
- If raw-video flipping behaves differently from these synthetic transformations, collect a small consented webcam development set and use it for calibration—not WLASL.